# Lab 23-01 · The Cost of Moving Data

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import memwall as mw


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · Wire energy in ngspice

*Predict* the source energy, the stored energy and the dissipated energy for 200 fF charged to 0.75 V, with a 1 kΩ and then a 10 kΩ driver.

In [2]:
print(f"predicted: supply {mw.switching_energy(200e-15, 0.75)*1e15:.2f} fJ, stored {0.5*200e-15*0.75**2*1e15:.2f} fJ")
s = ngspice("wire_energy.cir")
print({kk: s[kk] * 1e15 for kk in ("esup1", "estore", "ediss1", "esup2")}, "(fJ)")

predicted: supply 112.50 fJ, stored 56.25 fJ


{'esup1': 112.446, 'estore': 56.25, 'ediss1': 56.1959, 'esup2': 112.494} (fJ)


## Part 2 · Power budgets and pins

*Predict* the bandwidth a 60 W budget allows at 14.0 and 3.9 pJ/bit.

In [3]:
eb = np.linspace(1, 20, 200)
plt.figure()
plt.plot(eb, mw.max_bandwidth(60, eb * 1e-12) / 1e12)
for name, e in (("GDDR5", 14.0), ("HBM2", 3.9)):
    plt.plot(e, mw.max_bandwidth(60, e * 1e-12) / 1e12, "o")
    plt.annotate(name, (e, mw.max_bandwidth(60, e * 1e-12) / 1e12))
plt.xlabel("energy per bit (pJ)"); plt.ylabel("bandwidth in 60 W (TB/s)"); plt.show()
print(f"GDDR5 {mw.max_bandwidth(60, 14e-12)/1e9:.0f} GB/s, HBM2 {mw.max_bandwidth(60, 3.9e-12)/1e12:.2f} TB/s")
print(f"pin rate for 8 TB/s on 8,192 bits: {mw.per_pin_rate(8e12, 8192)/1e9:.2f} Gb/s")

GDDR5 536 GB/s, HBM2 1.92 TB/s
pin rate for 8 TB/s on 8,192 bits: 7.81 Gb/s


## Part 3 · Little's law

*Predict*: with 75,000 requests of 64 bytes outstanding, what bandwidth is left at 1.2 µs?

In [4]:
lat = np.linspace(200e-9, 2e-6, 200)
bw = 75_000 * 64 / lat
plt.figure(); plt.plot(lat * 1e9, bw / 1e12); plt.axhline(8, ls="--")
plt.xlabel("loaded latency (ns)"); plt.ylabel("deliverable bandwidth (TB/s)"); plt.show()
print(f"in flight for 8 TB/s at 600 ns: {mw.bytes_in_flight(8e12, 600e-9)/1e6:.1f} MB; at 1.2 us the 75,000 requests give {75_000*64/1.2e-6/1e12:.1f} TB/s")

in flight for 8 TB/s at 600 ns: 4.8 MB; at 1.2 us the 75,000 requests give 4.0 TB/s


## Part 4 · Decode with a KV cache

*Predict* the intensity limit at each context from §3.7 before plotting.

In [5]:
m = mw.Transformer(params=70e9, weight_bytes=1, layers=80, q_heads=64, kv_heads=8, head_dim=128, kv_bytes=1)
ctx = 32_768
b = m.max_batch(288e9, ctx)
print(f"KV per token {m.kv_per_token():.0f} B, per sequence {m.kv_per_sequence(ctx)/1e9:.3f} GB, batch {b}")
print(f"step {m.step_time(ctx, b, 8e12, 5e15)*1e3:.1f} ms, {m.tokens_per_second(ctx, b, 8e12, 5e15):.0f} tokens/s, "
      f"intensity {m.intensity(ctx, b):.1f}, limit {m.intensity_limit(ctx):.1f}, attention {m.attention_intensity():.0f} FLOP/B")
batch = np.arange(1, 513)
plt.figure()
for n in (4096, 32_768, 131_072):
    plt.semilogx(batch, [m.intensity(n, int(x)) for x in batch], label=f"{n//1024}k context")
plt.axhline(mw.ridge_point(5e15, 8e12), ls="--", label="ridge, 625 FLOP/B")
plt.xlabel("batch"); plt.ylabel("FLOP per byte"); plt.legend(); plt.show()

KV per token 163840 B, per sequence 5.369 GB, batch 40
step 35.6 ms, 1124 tokens/s, intensity 31.7, limit 42.1, attention 16 FLOP/B


### Halve the KV precision

*Predict* the new batch and intensity limit.

In [6]:
m4 = mw.Transformer(params=70e9, weight_bytes=1, layers=80, q_heads=64, kv_heads=8, head_dim=128, kv_bytes=0.5)
b4 = m4.max_batch(288e9, ctx)
print(f"4-bit KV: batch {b4}, limit {m4.intensity_limit(ctx):.1f} FLOP/B, step at batch {b} {m4.step_time(ctx, b, 8e12, 5e15)*1e3:.1f} ms "
      f"(was {m.step_time(ctx, b, 8e12, 5e15)*1e3:.1f} ms)")

4-bit KV: batch 81, limit 84.2 FLOP/B, step at batch 40 22.2 ms (was 35.6 ms)
